# Introduction to Video Understanding

## Instrucciones Generales

El siguiente práctico se puede realizar de manera **individual**. El formato de entregar es el **archivo .ipynb con todas las celdas ejecutadas**. Todas las preguntas deben ser respondida en celdas de texto. No se aceptará el _output_ de una celda de código como respuesta.

**Nombre:** COMPLETAR

**Fecha de entrega: 24 Agosto 2025**

El siguiente práctico cuanta con 2 secciones donde cada una contendrá 1 o más actividades a realizar. Algunas actividades correspondrán a escribir código y otras a responder preguntas.

**Importante.** Para facilitar su ejecución, cada sección puede ser ejecutada independientemente.

Se recomienda **fuertemente** revisar las secciones donde se entrega código porque algunas actividades de código pueden reutilizar el mismo código pero con cambios en algunas líneas.

## 1. Utilities

Essential libraries for executing this tutorial.

In [ ]:
!pip install imageio-ffmpeg

In [ ]:
import os, cv2
from tqdm.notebook import tqdm
import re
import math
import numpy as np
import torch.utils.data as data
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import time
import copy
import matplotlib.pyplot as plt
import pandas as pd

from PIL import Image
import torchvision

# from tensorflow_docs.vis import embed
# from moviepy.editor import *

## 2. Dataset

### Download data

We will use the [UCF11 dataset](https://www.crcv.ucf.edu/data/UCF_YouTube_Action.php) consisting of YouTube videos containing 11 action categories. For this tutorial, the dataset is hosted in a repository that we own.

In [ ]:
!wget --no-check-certificate https://www.crcv.ucf.edu/data/UCF11_updated_mpg.rar

In [ ]:
!unrar x '/content/UCF11_updated_mpg.rar'

### Exploring data

Let's explore the basic statistics of the dataset.

In [ ]:
def statistics(path_dataset):
  classes = os.listdir(path_dataset)
  num_classes = len(classes)
  dictionary = {}
  for name in classes:
    num_videos = len(os.listdir("{}/{}".format(path_dataset,name)))-1
    total_videos = len(os.listdir("{}/{}/Annotation".format(path_dataset,name)))
    dictionary[name] = {'unique_videos': num_videos, 'total_videos': total_videos}
  return dictionary

In [ ]:
path_dataset = "/content/UCF11_updated_mpg"
out = statistics(path_dataset)
pd.DataFrame.from_dict(out)

### Visualize Videos

Here we can visualize a selected video from the dataset.

In [ ]:
path_sample = "/content/UCF11_updated_mpg/basketball/v_shooting_04/v_shooting_04_01.mpg"

In [ ]:
import imageio
import os

# load the video file
clip = os.path.abspath(path_sample)

# create a function
def gifMkaker(inputPath, targetFormat, duration):
    # giving a output path
    outputPath = "animation" + targetFormat

    print(f'converting {inputPath} \n to {outputPath}')

    reader = imageio.get_reader(inputPath)
    # define fps of the gif
    # The keyword `fps` is no longer supported.
    # Use `duration`(in ms) instead, e.g. `fps=50` == `duration=20` (1000 * 1/50)
    fps = reader.get_meta_data()['duration']

    writer = imageio.get_writer(outputPath, duration=fps)

    for i, frames in enumerate(reader):
        writer.append_data(frames)
        # print(f'Frame{frames}')
        if i > duration:
          break
    print('done')
    writer.close()

gifMkaker(clip, '.gif', 30)

In [ ]:
from IPython.display import Image as ImageD
ImageD(filename="animation.gif")

### Preprocess data

Here we have two main functions for preprocess the videos.

The `vids_to_frames` function convert from video in mpg formart to N frames in jpg format.

In [ ]:
def vids_to_frames(path_videos, action, path):
    path_action = os.path.join(path_videos, action)
    list_cams = os.listdir(path_action)
    for cam in list_cams:
        path_cam_save = os.path.join(path, cam.split('.')[0])
        os.mkdir(path_cam_save)
        vid_path = os.path.join(path_action, cam)
        vidcap = cv2.VideoCapture(vid_path)
        success,image = vidcap.read()
        count = 0
        while success:
            path_cam_frame = os.path.join(path_cam_save, f'frame_{count}.jpg')
            cv2.imwrite(path_cam_frame, image)
            success,image = vidcap.read()
            count += 1

The `preprocess_data` function divides the data into train and val, and separate the videos into frames using the vids_to_frames function.

We need to define the `paths` and `split_size`.

In [ ]:
def preprocess_data(path_data, train_path, val_path, split_size):
    if not os.path.exists(train_path) and not os.path.exists(val_path):
        os.mkdir(train_path)
        os.mkdir(val_path)
        list_classes = os.listdir(path_data)
        for class_name in tqdm(list_classes):
            train_class = os.path.join(train_path, class_name)
            os.mkdir(train_class)
            val_class = os.path.join(val_path, class_name)
            os.mkdir(val_class)
            path_videos = os.path.join(path_data, class_name)
            list_actions = os.listdir(path_videos)
            list_actions = [a for a in list_actions if a != 'Annotation']
            num_actions = len(list_actions)
            num_actions_train = int(num_actions*split_size)
            action_train = list_actions[:num_actions_train]
            action_test = list_actions[num_actions_train:]
            for action in list_actions:
                if action in action_train:
                    vids_to_frames(path_videos, action, train_class)
                else:
                    vids_to_frames(path_videos, action, val_class)

Defining the paths to training data and validation data. Then we preprocess the whole dataset with `split_size=0.8`

In [ ]:
path_data = '/content/UCF11_updated_mpg'
train_path = '/content/train'
val_path = '/content/val'

In [ ]:
preprocess_data(path_data, train_path, val_path, 0.8) # it takes about ~4min

### Video dataset

Now, we will create our dataset for UCF11 dataset.



This is the Dataset Class created to transform and load the videos into the model using uniform temporal sampling. Note that `np.linspace` function allows to do that.

<figure>
<center>
<img src='https://www.sharpsightlabs.com/wp-content/uploads/2018/10/visual-representation-of-np-linspace-0-100-5.png' width="500" />
</center>
</figure>

In [ ]:
class VideoDataset(data.Dataset):

    def __init__(self, path_data, num_frames, transform=None):
        # This code loads all the videos and prepares the labels.
        self.list_classes = os.listdir(path_data)
        self.list_data = []
        self.list_labels = []
        for idx, c in enumerate(self.list_classes):
            path_class_data = os.path.join(path_data, c)
            class_data = os.listdir(path_class_data)
            self.list_data.extend(class_data)
            self.list_labels.extend([idx]*len(class_data))
        self.root_dir = path_data
        self.transform = transform
        self.num_frames = num_frames

    def __len__(self):
        # The number of all videos in the set
        return len(self.list_data)

    def natural_keys(self, text):
        return int(re.split(r'(\d+)', text)[1])

    def __getitem__(self, idx):
        # function to get the item or video i
        elem = self.list_data[idx]
        label = self.list_labels[idx]
        path_elem = os.path.join(self.root_dir, self.list_classes[label], elem)
        frames_elem = os.listdir(path_elem)
        frames_elem.sort(key=self.natural_keys)
        idx_frames = np.linspace(0, len(frames_elem)-1, num=self.num_frames, dtype = int)
        list_frames = []
        for ind_frame in idx_frames:
            frame_name = frames_elem[ind_frame]
            img_name = os.path.join(path_elem, frame_name)
            frame = Image.open(img_name).convert('RGB')
            list_frames.append(frame)
        process_data = self.transform(list_frames)
        return process_data, label

### Video transformation

These classes and functions are used to augment the video data, which is essential to train the model and avoid overfitting.

In [ ]:
class GroupRandomHorizontalFlip(object):
    """Randomly horizontally flips the given PIL.Image with a probability of 0.5
    """
    def __init__(self, is_flow=False):
        self.is_flow = is_flow

    def __call__(self, img_group, is_flow=False):
        v = random.random()
        if v < 0.5:
            ret = [img.transpose(Image.FLIP_LEFT_RIGHT) for img in img_group]
            if self.is_flow:
                for i in range(0, len(ret), 2):
                    ret[i] = ImageOps.invert(ret[i])  # invert flow pixel values when flipping
            return ret
        else:
            return img_group

In [ ]:
class GroupMultiScaleCrop(object):

    def __init__(self, input_size, scales=None, max_distort=1, fix_crop=True, more_fix_crop=True):
        self.scales = scales if scales is not None else [1, .875, .75, .66]
        self.max_distort = max_distort
        self.fix_crop = fix_crop
        self.more_fix_crop = more_fix_crop
        self.input_size = input_size if not isinstance(input_size, int) else [input_size, input_size]
        self.interpolation = Image.BILINEAR

    def __call__(self, img_group):

        im_size = img_group[0].size

        crop_w, crop_h, offset_w, offset_h = self._sample_crop_size(im_size)
        crop_img_group = [img.crop((offset_w, offset_h, offset_w + crop_w, offset_h + crop_h)) for img in img_group]
        ret_img_group = [img.resize((self.input_size[0], self.input_size[1]), self.interpolation)
                         for img in crop_img_group]
        return ret_img_group

    def _sample_crop_size(self, im_size):
        image_w, image_h = im_size[0], im_size[1]

        # find a crop size
        base_size = min(image_w, image_h)
        crop_sizes = [int(base_size * x) for x in self.scales]
        crop_h = [self.input_size[1] if abs(x - self.input_size[1]) < 3 else x for x in crop_sizes]
        crop_w = [self.input_size[0] if abs(x - self.input_size[0]) < 3 else x for x in crop_sizes]

        pairs = []
        for i, h in enumerate(crop_h):
            for j, w in enumerate(crop_w):
                if abs(i - j) <= self.max_distort:
                    pairs.append((w, h))

        crop_pair = random.choice(pairs)
        if not self.fix_crop:
            w_offset = random.randint(0, image_w - crop_pair[0])
            h_offset = random.randint(0, image_h - crop_pair[1])
        else:
            w_offset, h_offset = self._sample_fix_offset(image_w, image_h, crop_pair[0], crop_pair[1])

        return crop_pair[0], crop_pair[1], w_offset, h_offset

    def _sample_fix_offset(self, image_w, image_h, crop_w, crop_h):
        offsets = self.fill_fix_offset(self.more_fix_crop, image_w, image_h, crop_w, crop_h)
        return random.choice(offsets)

    @staticmethod
    def fill_fix_offset(more_fix_crop, image_w, image_h, crop_w, crop_h):
        w_step = (image_w - crop_w) // 4
        h_step = (image_h - crop_h) // 4

        ret = list()
        ret.append((0, 0))  # upper left
        ret.append((4 * w_step, 0))  # upper right
        ret.append((0, 4 * h_step))  # lower left
        ret.append((4 * w_step, 4 * h_step))  # lower right
        ret.append((2 * w_step, 2 * h_step))  # center

        if more_fix_crop:
            ret.append((0, 2 * h_step))  # center left
            ret.append((4 * w_step, 2 * h_step))  # center right
            ret.append((2 * w_step, 4 * h_step))  # lower center
            ret.append((2 * w_step, 0 * h_step))  # upper center

            ret.append((1 * w_step, 1 * h_step))  # upper left quarter
            ret.append((3 * w_step, 1 * h_step))  # upper right quarter
            ret.append((1 * w_step, 3 * h_step))  # lower left quarter
            ret.append((3 * w_step, 3 * h_step))  # lower righ quarter

        return ret

In [ ]:
class GroupScale(object):
    """ Rescales the input PIL.Image to the given 'size'.
    'size' will be the size of the smaller edge.
    For example, if height > width, then image will be
    rescaled to (size * height / width, size)
    size: size of the smaller edge
    interpolation: Default: PIL.Image.BILINEAR
    """

    def __init__(self, size, interpolation=Image.BILINEAR):
        self.worker = torchvision.transforms.Scale(size, interpolation)

    def __call__(self, img_group):
        return [self.worker(img) for img in img_group]


class GroupCenterCrop(object):
    def __init__(self, size):
        self.worker = torchvision.transforms.CenterCrop(size)

    def __call__(self, img_group):
        return [self.worker(img) for img in img_group]


class Stack(object):

    def __init__(self, roll=False):
        self.roll = roll

    def __call__(self, img_group):
        if img_group[0].mode == 'L':
            return np.concatenate([np.expand_dims(x, 2) for x in img_group], axis=2)
        elif img_group[0].mode == 'RGB':
            if self.roll:
                return np.concatenate([np.array(x)[:, :, ::-1] for x in img_group], axis=2)
            else:
                return np.concatenate(img_group, axis=2)


class ToTorchFormatTensor(object):
    """ Converts a PIL.Image (RGB) or numpy.ndarray (H x W x C) in the range [0, 255]
    to a torch.FloatTensor of shape (C x H x W) in the range [0.0, 1.0] """

    def __init__(self, div=True):
        self.div = div

    def __call__(self, pic):
        if isinstance(pic, np.ndarray):
            # handle numpy array
            img = torch.from_numpy(pic).permute(2, 0, 1).contiguous()
        else:
            # handle PIL Image
            img = torch.ByteTensor(torch.ByteStorage.from_buffer(pic.tobytes()))
            img = img.view(pic.size[1], pic.size[0], len(pic.mode))
            # put it from HWC to CHW format
            # yikes, this transpose takes 80% of the loading time/CPU
            img = img.transpose(0, 1).transpose(0, 2).contiguous()
        return img.float().div(255) if self.div else img.float()


class GroupNormalize(object):
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        rep_mean = self.mean * (tensor.size()[0] // len(self.mean))
        rep_std = self.std * (tensor.size()[0] // len(self.std))

        # TODO: make efficient
        for t, m, s in zip(tensor, rep_mean, rep_std):
            t.sub_(m).div_(s)

        return tensor


def get_transform():
    cropping = torchvision.transforms.Compose([
        GroupScale(256),
        GroupCenterCrop(224),
    ])
    transform = torchvision.transforms.Compose([
        cropping,
        Stack(roll=False),
        ToTorchFormatTensor(div=True),
        GroupNormalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return transform

def scale_size(input_size):
    return input_size * 256 // 224

## 3. Dataloader

### Instance transformation

Instantiate the transformations for training and validation.
It is necessary to define a fixed `input_size`, and normalization parameters (`input_mean` and `input_std`).

In [ ]:
# size of frame
input_size = 224
# normalization
input_mean = [0.485, 0.456, 0.406]
input_std = [0.229, 0.224, 0.225]
# backbone name
arch = "Resnet"

In [ ]:
transform_train = torchvision.transforms.Compose([
                                                  GroupMultiScaleCrop(input_size, [1, .875, .75, .66]),
                                                  GroupRandomHorizontalFlip(is_flow=False),
                                                  Stack(roll=(arch in ['BNInception', 'InceptionV3'])),
                                                  ToTorchFormatTensor(div=(arch not in ['BNInception', 'InceptionV3'])),
                                                  GroupNormalize(input_mean, input_std)
                                                  ])

In [ ]:
transform_val = torchvision.transforms.Compose([
                       GroupCenterCrop(input_size),
                       Stack(roll=(arch in ['BNInception', 'InceptionV3'])),
                       ToTorchFormatTensor(div=(arch not in ['BNInception', 'InceptionV3'])),
                       GroupNormalize(input_mean, input_std)
                   ])

### Instance video dataset

Instantiate the VideoDataset for training and validation with 8 frames per video

In [ ]:
num_frames = 5

In [ ]:
dataset_train = VideoDataset('/content/train', num_frames, transform_train)
dataset_val = VideoDataset('/content/val', num_frames, transform_val)

## 4. Create model

This is the model used to classify the videos. It is based on a pre-trained ResNet-34 backbone and avg pooling to reduce the temporal dimension.

We are going to build the two versions we explored in class. `VideoCNN`, which is a model based solely on CNN without temporal processing, and `VideoRNN`, which is a model based on CNN and RNN to process visual and temporal information.

In [ ]:
class VideoCNN(nn.Module):
    def __init__(self, num_classes, num_frames=8):
        super(VideoCNN, self).__init__()
        # create a pre-trained cnn
        self.net = models.resnet34(pretrained=True)
        # create a classifier
        self.fc = nn.Linear(in_features=self.net.fc.in_features,
                            out_features=num_classes)
        self.net.fc = nn.Identity()
        self.num_segments = num_frames

    def forward(self, inputs):
        # flatten input
        inputs = inputs.view((-1, 3) + inputs.size()[-2:])
        # compute frames representation with cnn
        out = self.net(inputs)
        # unflatten output of cnn
        out = out.view((-1, self.num_segments) + out.size()[1:])
        # average pooling of representations
        out = torch.mean(out, dim = 1)
        # classify
        out = self.fc(out)
        return out

In [ ]:
class VideoRNN(nn.Module):
    def __init__(self, num_classes, num_frames=8):
        super(VideoRNN, self).__init__()
        # create a pre-trained cnn
        self.net = models.resnet34(pretrained=True)
        # create a rnn (1 layer)
        self.rnn = nn.GRU(input_size=self.net.fc.in_features,
                          hidden_size=self.net.fc.in_features,
                          num_layers=1,
                          batch_first=True)
        self.dropout= nn.Dropout(0.1)
        # create a classifier
        self.fc = nn.Linear(in_features=self.net.fc.in_features,
                            out_features=num_classes)
        self.net.fc = nn.Identity()
        self.num_segments = num_frames

    def forward(self, inputs):
        # flatten input
        inputs = inputs.view((-1, 3) + inputs.size()[-2:])
        # compute representation with cnn
        out = self.net(inputs)
        # unflatten output of cnn
        out = out.view((-1, self.num_segments) + out.size()[1:])
        # processing frames representations with rnn
        bz, seq, dim = out.shape
        h0 = torch.randn(1, bz, dim, device=inputs.device)
        out, hn = self.rnn(out, h0)
        # classify using the last frame representation
        out = self.dropout(out[:,-1])
        out = self.fc(out)
        return out

### Instance model

Instantiate the model. We can decide to use `VideoCNN` or `VideoRNN`.

In [ ]:
num_classes = len(dataset_train.list_classes)
model_type = "cnn+rnn"

if model_type == "cnn+rnn":
  model = VideoRNN(num_classes, num_frames)
elif model_type == "cnn":
  model = VideoCNN(num_classes, num_frames)

## 5. Train the model

Train the model during 12 epoch

In [ ]:
num_epochs = 3

In [ ]:
device = torch.device("cuda")
model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [ ]:
batch_size = 40

trainloader = torch.utils.data.DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
testloader = torch.utils.data.DataLoader(dataset_val, batch_size=batch_size, shuffle=False)

dataloaders={'train': trainloader, 'val':testloader}
dataset_sizes = {'train': len(dataset_train), 'val': len(dataset_val)}

In [ ]:
def train_model(model, criterion, optimizer, num_epochs=25):
    since = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    dict_acc = {'train': [], 'val': []}
    dict_loss = {'train': [], 'val': []}
    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0
            num_elem = 0

            # Iterate over data.
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # zero the parameter gradients
                optimizer.zero_grad()

                # forward
                # track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    preds = torch.argmax(outputs, dim = 1)
                    loss = criterion(outputs, labels)

                    # backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                num_elem += inputs.size(0)
            # if phase == 'train':
            #     scheduler.step()

            epoch_loss = running_loss / num_elem
            epoch_acc = running_corrects.double() / num_elem

            dict_acc[phase].append(epoch_acc)
            dict_loss[phase].append(epoch_loss)
            print('{} Loss: {:.4f} Acc: {:.4f}'.format(
                phase, epoch_loss, epoch_acc))

            # deep copy the model
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(
        time_elapsed // 60, time_elapsed % 60))
    print('Best val Acc: {:4f}'.format(best_acc))

    # load best model weights
    model.load_state_dict(best_model_wts)
    return model, dict_acc, dict_loss

In [ ]:
model, dict_acc, dict_loss = train_model(model, criterion, optimizer, num_epochs=num_epochs)

## 6. Train Graphs

In [ ]:
def parse_data(data):
    new_data = []
    for d in data:
        new_data.append(d.item())
    return new_data

In [ ]:
epochs = list(range(num_epochs))
plt.plot(epochs, parse_data(dict_acc['train']), 'r', label='Train')
plt.plot(epochs, parse_data(dict_acc['val']), 'b', label='Val')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.show()

## 7. Positive and Negative examples

In [ ]:
def getInferenceModel(model, testloader):
    for inputs, labels in testloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        preds = torch.argmax(outputs, dim = 1)
        break
    return preds, inputs, labels

In [ ]:
def plot_example(dataset, idx, pred):

    fig=plt.figure(figsize=(20, 8))
    columns = 4
    rows = 2
    elem = dataset.list_data[idx]
    label = dataset.list_labels[idx]
    fig.suptitle('GT: {}, P: {}'.format(dataset.list_classes[label], dataset.list_classes[pred]), fontsize=16)
    path_elem = os.path.join(dataset.root_dir, dataset.list_classes[label], elem)
    frames_elem = os.listdir(path_elem)
    frames_elem.sort(key=dataset.natural_keys)
    idx_frames = np.linspace(0, len(frames_elem)-1, num=dataset.num_frames, dtype = int)
    list_frames = []
    for i, ind_frame in enumerate(idx_frames):
        frame_name = frames_elem[ind_frame]
        img_name = os.path.join(path_elem, frame_name)
        frame = Image.open(img_name).convert('RGB')
        ax = fig.add_subplot(rows, columns, i+1)
        ax.set_title("t = {}".format(ind_frame))
        ax.axis('off')
        plt.imshow(frame)

In [ ]:
model.eval()
testloader = torch.utils.data.DataLoader(dataset_val, batch_size=30, shuffle=False)

In [ ]:
preds, inputs, labels = getInferenceModel(model, testloader)

In [ ]:
torch.cuda.empty_cache()

In [ ]:
good_examples = np.where((preds == labels).cpu() == True)[0]
good_examples

In [ ]:
bad_examples = np.where((preds != labels).cpu() == True)[0]
bad_examples

In [ ]:
idx = good_examples[-1]
plot_example(dataset_val, idx, preds[idx])

In [ ]:
idx = bad_examples[-1]
plot_example(dataset_val, idx, preds[idx])

## 8. Activity


In this section, you will be asked about the topics studied in the theoretical and practical class.

1. What are the main challenges of video analysis?

Write the answer here ...

2. Why is video analysis important?

Write the answer here ...

3. What is action classification?

Write the answer here ...

4. Mention and describe the temporal sampling used in this tutorial

Write the answer here ...

5. What is the temporal pooling followed in this lab? What are the problems with it?

Write the answer here ...

6. Train and evaluate the model with 4 frames per video instead of 8 frames. How does the model performance be affected? Note: Use code of section 6 to show the training graph.

In [ ]:
## Write the code here ...

Write the answer here ...